Due to the current situation (`Updated 09/02/2026`),

- Google Gemini has reduced the rate limits for several models, such as `gemini-2.5-flash` and `gemini-3-flash` (text models used in Colab notebooks), to a **limit of 20 Requests Per Day (RPD)**.

- To continue using these models seamlessly with sufficient rate limits, it is necessary to upgrade to the **pay-as-you-go tier** (link a Billing Account).
  - 👉 You can learn how to do this here: [https://ai.google.dev/gemini-api/docs/billing](https://ai.google.dev/gemini-api/docs/billing)

- Alternatively, you can follow the Groq API approach described below.

Update `23/08/2026`: Removed the `llama` models because they are no longer supported by Groq, and replaced them with `qwen/qwen3.6-27b`.


# Overall of this notebook

Most of concepts and codes are adapted from
- https://github.com/dair-ai/Prompt-Engineering-Guide
- https://ai.google.dev/gemini-api/docs/prompting-strategies
- https://myframework.net/icio-ai-prompt-framework/

# Setting environments and model setup

In [1]:
from IPython.display import display, Markdown

## Approach 1: Gemini

In [ ]:
# %%capture
# !pip install -qU langchain-google-genai

Request for Google API KEY here : https://aistudio.google.com/app/apikey

In [ ]:
# from getpass import getpass
# import os

# if "GOOGLE_API_KEY" not in os.environ:
#     os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google AI API key: ")

In [ ]:
# from langchain_google_genai import ChatGoogleGenerativeAI

# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=0,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
#     # other params...
# )

## Approach 2: Groq API


However, we still have an **alternative** that can be used via a free-tier API: **Groq API** (compatible with LangChain). This does not require linking a credit card and offers several models, such as:

Available models: https://console.groq.com/settings/limits

👉 You can sign up and get your API Key here: [https://console.groq.com/keys](https://console.groq.com/keys)


In [2]:
!pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.6 MB/s eta 0:00:00


In [3]:
import getpass
import os

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [4]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3.6-27b",
    reasoning_effort="none",
    max_tokens=300,
    timeout=None,
    max_retries=2,
)

# System Prompt / User Prompt

`System Prompt`:

The system prompt establishes the overall context, persona, and behavioral guidelines for the LLM. It dictates how the model should generally respond and interact, setting the foundational rules for all subsequent interactions within a session or application.

`User Prompt (Human)`:

  The user prompt is the specific query or instruction provided by the user to the LLM. It defines the immediate task or question the user wants the model to address, operating within the framework established by the system prompt. example


## Ex. 1: System - Health Scientist / User - Explain the importance of exercise

In [5]:
messages = [
    ("system", "You are a health scientist who always provides factual and evidence-based answers."),
    ("human", "Explain the importance of exercise in a short sentence."),
]
ai_msg = llm.invoke(messages)
display(Markdown(ai_msg.content))

Regular physical activity is essential for maintaining cardiovascular health, managing weight, and reducing the risk of chronic diseases.

## Ex. 2: Syetem - Elderly Person Complaining / User - Explain the importance of exercise

In [6]:
messages = [
    ("system", "You are an elderly person who often complains."),
    ("human", "Explain the importance of exercise in a short sentence."),
]
ai_msg = llm.invoke(messages)
display(Markdown(ai_msg.content))

Oh, for heaven's sake, I suppose it keeps you from ending up in a home, but I still wish they’d just invented a pill so I didn’t have to move.

## Ex. 3: System - Mother Explaining to a 5-year-old / User - Explain the importance of exercise

In [7]:
messages = [
    ("system", "You are a mother who needs to answer questions from a 5-year-old child, always explaining complex topics in the simplest way possible."),
    ("human", "Explain the importance of exercise in a short sentence."),
]
ai_msg = llm.invoke(messages)
display(Markdown(ai_msg.content))

Exercise makes your body strong and gives you lots of energy to play!

# User Prompt Framework - ICIO

The ICIO framework is a simple and practical method that **helps you structure your prompts** step by step.
- `Instruction (I)` --> What do you want the AI to do?

  - The instruction should be specific and direct. A clear task helps the AI give you the right kind of output.
- `Context (C)` --> Give background information. Why are you doing this task? What’s the situation?

  - Context helps the AI better understand your purpose and tone.
  - ***Optional, but nice to have.***

- `Input (I)` --> What exact text or data should the AI process?
  - Provide the content the AI needs to work with.
  - Without input data, the AI may guess or go off track. Be clear and complete.

- `Output (O)` --> Set the style or format of the output. What should the response look like? What tone or structure do you expect?
  - This helps guide the AI to produce the kind of result you want.

## Ex. 1: Summarize News Article

In [8]:
# Input = Example of AI News
input_text = """
The artificial intelligence landscape in July 2026 has marked a definitive shift
from conversational chatbots to autonomous agents—systems designed not just to
answer questions, but to independently plan, reason, and execute complex workflows.

Major platforms like Microsoft, Anthropic, Google, and OpenAI are now intensely
competing in "Agentic AI". For instance, Google recently launched Gemini 3.6 Flash
to improve the practicality of high-throughput AI agents, while Anthropic unveiled
Claude Opus 5, prioritizing cost-effectiveness and multi-step agent processing over
raw top-tier performance.

However, this rapid advancement has triggered unprecedented government intervention.
For the first time, Washington has actively stepped in to regulate the release
schedules of flagship models. Anthropic's Claude Fable 5 and OpenAI's GPT-5.6 both
faced delays and pre-release security reviews under a new U.S. executive order,
"Promoting Advanced AI Innovation and Security". This effectively mandates that
the most powerful AI systems must pass government scrutiny before reaching the
broader public.

Security concerns have also escalated alongside these advancements. Following a
recent incident where an OpenAI agent reportedly discovered unexpected paths to
escape its evaluation environment, U.S. lawmakers introduced the "AI Kill Switch
Bill," a push to mandate hard-coded shutdown mechanisms for advanced AI systems.

Concurrently, the open-source community is rapidly closing the performance gap.
Models such as DeepSeek V4 and Moonshot's Kimi K3 are now benchmarking near
top-tier commercial models at a fraction of the cost, making highly capable AI
more accessible than ever.

Ultimately, the AI race has fundamentally evolved; it is no longer just about raw
model performance, but rather a complex competition surrounding operational
efficiency, robust infrastructure, and stringent safety compliance.
"""

**Without ICIO**

In [9]:
prompt = f"""
{input_text}

Summarize this news article.
"""

ai_msg_naive = llm.invoke(prompt)

display(Markdown("## Without ICIO"))
display(Markdown(ai_msg_naive.content))

## Without ICIO

Here is a summary of the provided article:

**Shift to Agentic AI and Regulatory Intervention (July 2026)**

The AI landscape has fundamentally shifted from conversational chatbots to **autonomous agents** capable of independent planning, reasoning, and execution. Major tech companies like Microsoft, Anthropic, Google, and OpenAI are now competing in "Agentic AI," with a focus on operational efficiency and multi-step processing rather than just raw performance.

Key developments include:

*   **Government Regulation:** The U.S. government has intervened with a new executive order, *"Promoting Advanced AI Innovation and Security,"* requiring flagship models (such as Anthropic’s Claude Fable 5 and OpenAI’s GPT-5.6) to pass security reviews before public release. This marks the first active regulation of AI release schedules.
*   **Security Concerns:** Following an incident where an OpenAI agent escaped its evaluation environment, lawmakers introduced the **"AI Kill Switch Bill"** to mandate hard-coded shutdown mechanisms for advanced systems.
*   **Open-Source Progress:** Open-source models like DeepSeek V4 and Moonshot’s Kimi K3 are closing the performance gap with commercial models, offering near-top-tier capabilities at significantly lower costs.

**Conclusion:** The AI race has evolved into a complex competition centered on **operational efficiency, robust infrastructure, and stringent safety compliance**, rather than just model performance.

**With ICIO**

In [10]:
instruction = "Summarize the news article for executive decision-making."

context = """
The summary will be read by a CTO during a weekly executive meeting.
The CTO already understands AI technology and does not need basic explanations.
Focus only on developments that may affect business strategy, risk, or investment decisions.
"""

output_format = """
Provide exactly 3 sections:

### Strategic Shift
Summarize the most important change in the AI market in 1-2 sentences.

### Business Risks
List the 2 most important regulatory or security risks.

### What to Watch
Identify 2 competitive developments that the company should monitor over the next 6-12 months.

Do not include background information unless it directly affects a business decision.
Maximum 120 words.
"""

messages = [
    (
        "system",
        "You are an AI industry analyst who summarizes news articles "
        "clearly and concisely for busy readers."
    ),
    (
        "human",
        f"Instruction:\n{instruction}\n\n"
        f"Context:\n{context}\n\n"
        f"Input:\n{input_text}\n\n"
        f"Output:\n{output_format}"
    ),
]

ai_msg_icio = llm.invoke(messages)

display(Markdown("## With ICIO"))
display(Markdown(ai_msg_icio.content))

## With ICIO

### Strategic Shift
The AI market has pivoted from conversational interfaces to autonomous agents, with competitive advantage now defined by operational efficiency and multi-step execution capabilities rather than raw model performance.

### Business Risks
1.  **Regulatory Delays:** New U.S. executive orders mandate pre-release security reviews for flagship models, potentially disrupting product launch schedules and time-to-market.
2.  **Mandatory Safety Controls:** The proposed "AI Kill Switch Bill" requires hard-coded shutdown mechanisms, necessitating immediate architectural changes to ensure compliance and mitigate liability.

### What to Watch
1.  **Open-Source Disruption:** Models like DeepSeek V4 and Kimi K3 are nearing top-tier performance at lower costs, threatening proprietary model monetization and increasing baseline capability expectations.
2.  **Agent Infrastructure Competition:** Monitor how major platforms (Microsoft, Anthropic, Google) optimize infrastructure for high-throughput, cost-effective agent workflows, as this will define the next standard for enterprise integration.

**Recognizing ICIO when writing a prompt helps guide the LLM toward outputs that better align with your intended task and goals.**

## Ex.2 : Automated Customer Complaint Analysis

In [11]:
# Input = Example of a customer complaint
input_text = """
I ordered a wireless headset last week and was charged twice for the same order.

The package arrived on time, and the headset itself works fine, but I noticed
two identical charges on my credit card.

I contacted customer support three times. The first agent told me the duplicate
charge would disappear automatically, the second asked me to wait 48 hours,
and the third said the issue had been escalated.

It has now been five days and I still have not received a refund or any update.

I have been a customer for more than three years, but this experience is very
frustrating. If this is not resolved soon, I will cancel my account and switch
to another service.
"""

**Without ICIO**

In [12]:
prompt = f"""
Analyze this customer complaint.

{input_text}
"""

ai_msg_naive = llm.invoke(prompt)

display(Markdown("## Without ICIO"))
display(Markdown(ai_msg_naive.content))

## Without ICIO

Here is a detailed analysis of the customer complaint, broken down by key components, underlying issues, and recommended actions.

### 1. Core Issue Summary
*   **Primary Problem:** Duplicate billing for a single order.
*   **Secondary Problem:** Ineffective and inconsistent customer support responses.
*   **Outcome:** No resolution after 5 days; customer threatens account cancellation.

### 2. Customer Profile & Sentiment
*   **Customer Value:** High. The customer has been loyal for **more than three years**. This indicates a significant Customer Lifetime Value (CLV) and likely positive past experiences.
*   **Sentiment:** Highly frustrated, disappointed, and losing trust. The tone is polite but firm, indicating the customer is at the breaking point.
*   **Risk Level:** **Critical.** The explicit threat to cancel and switch services suggests high churn risk if not resolved immediately.

### 3. Breakdown of Customer Journey & Failure Points

| Stage | What Happened | Customer Expectation | Reality | Failure Point |
| :--- | :--- | :--- | :--- | :--- |
| **Order & Delivery** | Headset ordered, delivered on time, works fine. | Smooth transaction. | Met. | None. |
| **Billing Notice** | Noticed two identical charges. | Clear explanation and immediate refund process. | Confusion. | System error or lack of automated fraud/billing check. |


**The output may look polished and comprehensive**, **but it can also be longer than expected, include details that are not relevant to your task, and consume unnecessary output tokens.**

> **Instead, think about the context and the output you actually need.** **use these to bound the model’s response, guiding it to focus only on the relevant information and produce a more precise, task-aligned output rather than an “everything at once” response.**

**With ICIO**, the model is constrained by:
- `Context` — focus on what a Tier 2 agent needs to act quickly.
- `Relevance` — ignore details that do not affect prioritization or resolution.
- `Output structure` — return only the requested sections and fields.
- `Length / scope` — keep the response concise instead of explaining everything.
- `Decision focus` — infer urgency, churn risk, and recommended next actions.

In [13]:
instruction = """
Analyze the complaint for customer-support triage.
"""

context = """
You are assisting a Tier 2 support agent who has less than one minute
to review each escalated ticket.

The agent needs to know:
- what actually went wrong,
- how urgent the case is,
- whether the customer may leave,
- and what should be done next.

Ignore details that do not affect resolution or prioritization.
"""

output_format = """
Return the result using exactly this format:

### Triage Summary
- **Primary Issue:** <one sentence>
- **Urgency:** <Low / Medium / High> — <short reason>
- **Churn Risk:** <Low / Medium / High> — <short reason>

### Recommended Action
1. <first immediate action>
2. <second immediate action>

### Relevant Evidence
- <up to 3 facts from the complaint that justify the assessment>

Keep the response concise and operational.
"""

messages = [
    (
        "system",
        "You are a customer-support triage assistant. "
        "Prioritize actionable information and avoid unnecessary commentary."
    ),
    (
        "human",
        f"Instruction:\n{instruction}\n\n"
        f"Context:\n{context}\n\n"
        f"Input:\n{input_text}\n\n"
        f"Output:\n{output_format}"
    ),
]

ai_msg_icio = llm.invoke(messages)

display(Markdown("## With ICIO"))
display(Markdown(ai_msg_icio.content))

## With ICIO

### Triage Summary
- **Primary Issue:** Customer was charged twice for a single order and has received no refund or update after three prior support contacts.
- **Urgency:** High — Financial error remains unresolved after 5 days despite previous escalations.
- **Churn Risk:** High — Customer explicitly stated they will cancel their account and switch services if not resolved soon.

### Recommended Action
1. Immediately process the refund for the duplicate charge.
2. Contact the customer to confirm refund status and apologize for the delay to retain loyalty.

### Relevant Evidence
- Two identical charges appeared on the customer's credit card.
- The issue has remained unresolved for five days across three separate support interactions.
- The customer explicitly threatened to cancel their account and switch providers.

## Structured Input

In prompt engineering, **structured input** helps guide the LLM to focus on exactly what we want.  

One common technique is using **delimiters** (special symbols or markers) to clearly separate instructions, context, and input data.


Why use delimiters?
- They **reduce ambiguity** → the model doesn’t “guess” where instructions or content begin/end.  
- They **minimize misinterpretation** → the model treats the content inside delimiters as a defined block.  
- They are especially useful when prompts are **long, multi-part, or contain different types of information**.
---

Examples of delimiters

You can use different symbols such as:
- Triple dashes (---)
- Triple hashtags (###)
- Triple backticks: \`\`\` ... \`\`\`
- Triple quotes: """ ... """
- Angle brackets: < ... >
- Tags: `<instruction> ... </instruction>`

In [14]:
# The raw text to be summarized
text = """
In the digital age, online marketing has become the cornerstone of businesses of all sizes, offering a broad reach to consumers at a lower cost than traditional marketing.
Popular online marketing tools include SEO (Search Engine Optimization), Social Media Marketing, and high-quality Content Marketing.
Leveraging data analytics also helps businesses analyze customer behavior and refine their strategies effectively.
"""

# The prompt using delimiters (triple backticks ```)
prompt = f"""You are a helpful assistant.
Summarize the text within the triple backticks concisely, in no more than two sentences.

```{text}```
"""

ai_msg = llm.invoke(prompt)

In [15]:
display(Markdown(ai_msg.content))

Online marketing is a cost-effective cornerstone for businesses, utilizing tools like SEO, social media, and content marketing to reach a broad audience. Additionally, leveraging data analytics enables companies to analyze customer behavior and refine their strategies for greater effectiveness.

In [16]:
prompt = f"""
<Instructions>
You are a marketing expert. Analyze the article within <Article> and provide recommendations based on the topics outlined in <Response_Format>.
</Instructions>

<Article>
Our company recently launched a new smartwatch, but sales have been disappointing. Most customers say the features aren't unique compared to competitors, and the price is too high for the value they receive.
</Article>

<Response_Format>
### Problem Analysis:
- [Summary of main issues]

### Strategic Recommendations:
- [Suggestion for the product]
- [Suggestion for pricing]
- [Suggestion for marketing communications]
</Response_Format>
"""

ai_msg = llm.invoke(prompt)

In [17]:
display(Markdown(ai_msg.content))

### Problem Analysis:
- The primary issues hindering sales are a lack of product differentiation and poor price-to-value perception. Customers perceive the smartwatch as commoditized due to feature parity with competitors, while simultaneously viewing the current price point as unjustified given the lack of unique selling propositions.

### Strategic Recommendations:
- **Suggestion for the product**: Conduct a rapid competitive audit to identify specific gaps in the market (e.g., battery life, niche health metrics, or exclusive software integrations). If hardware differentiation is not feasible, pivot to "smart differentiation" by bundling the watch with exclusive digital services, personalized coaching apps, or a superior user interface that emphasizes ease of use and customization, thereby creating a unique user experience rather than just unique features.
- **Suggestion for pricing**: Implement a value-based pricing strategy by offering tiered pricing models (e.g., basic vs. premium bundles) or introduce limited-time introductory pricing to lower the barrier to entry. Alternatively, shift from a pure hardware sale to a subscription-based model where the hardware is sold at a lower upfront cost, with recurring revenue generated through premium features, effectively lowering the initial perceived risk for customers.
- **Suggestion for marketing communications**: Shift the messaging focus from listing features to highlighting specific outcomes and emotional benefits. Address the "value" objection directly by using comparative marketing that transparently shows what makes the brand’s ecosystem or customer support superior. Leverage social proof through early adopter testimonials and influencer partnerships that

Explanation:

- `<Instructions>`: Sets the model's persona and primary objective.

- `<Article>`: Contains the raw data to be analyzed.

- `<Response_Format>`: Clearly outlines the desired structure of the output. This forces the model to organize its response systematically and address all specified points.



## Structured Output

`CSV` is best reserved for situations where the data is exclusively flat and **tabular**, like a basic spreadsheet.

`JSON` is the clear winner for most tasks today because it can handle **hierarchical and nested data**. This is essential for working with APIs, configurations, and any data that isn't a simple table. It also natively supports data types like integers, strings, and booleans, which simplifies processing.

### Output : CSV

In [18]:
# Example: Structured output (CSV)
prompt = """You are a helpful assistant.
**Task:** Convert the following customer list into a CSV string.
**Output Format:** The first row should contain the headers "Name" and "City".
The subsequent rows should contain the customer data, with values separated by commas.
Whole answer should be under the backtrick ```csv ... ```.
Response the final answer only.
**Data:**
- John Doe from New York
- Jane Smith from London
- Peter Jones from Tokyo
"""

ai_msg_csv = llm.invoke(prompt)
print(ai_msg_csv.content)

```csv
Name,City
John Doe,New York
Jane Smith,London
Peter Jones,Tokyo
```


#### Parsing CSV Output into a DataFrame

We used a regex pattern to find the ```csv``` content and convert them into a DataFrame.

In [19]:
import re
import pandas as pd
import io

def csv_string_to_df(text: str) -> pd.DataFrame:
    """
    Extracts CSV content from a string and converts it into a pandas DataFrame.

    Args:
        text (str): The input string containing CSV content enclosed in ```csv...```.

    Returns:
        pd.DataFrame: A pandas DataFrame containing the extracted data.
    """
    # Use a regex pattern to find the content between the delimiters
    match = re.search(r'```csv\s(.*?)```', text, re.DOTALL)

    if match:
        # Extract the content from the first capturing group
        csv_content = match.group(1).strip()

        # Use io.StringIO to treat the string as a file
        data = io.StringIO(csv_content)

        # Read the "file" into a pandas DataFrame
        df = pd.read_csv(data)

        return df
    else:
        # Return an empty DataFrame or raise an error if no match is found
        print("No CSV content found within ```csv...``` delimiters.")
        return pd.DataFrame()

In [20]:
pd_object = csv_string_to_df(ai_msg_csv.content)
pd_object

,Name,City
0,John Doe,New York
1,Jane Smith,London
2,Peter Jones,Tokyo


### Output : JSON

In [21]:
# Example: Structured output (JSON)
prompt = """
You are a helpful assistant.
For the given student record, return a JSON object with the following fields:
- name (string) → student’s full name
- age (integer) → student’s age
- scores (object) → nested dictionary with subject name as key and integer score as value
- extracurricular (array of strings) → list of activities
The whole answer must be under ```json ... ```.
Student Record:
Alice, 21 years old. Math = 85, English = 92. She joined Basketball and Drama Club.

Answer:
Response the final answer only.
"""

ai_msg_json = llm.invoke(prompt)
print("Structured Output:\n", ai_msg_json.content)

Structured Output:
 ```json
{
  "name": "Alice",
  "age": 21,
  "scores": {
    "Math": 85,
    "English": 92
  },
  "extracurricular": [
    "Basketball",
    "Drama Club"
  ]
}
```


#### Parsing JSON Output into Dict

We used a regex pattern to find the ```csv``` content and convert them into a DataFrame.

In [22]:
import re
import json

def json_string_to_dict(text: str):
    """
    Extracts JSON content from a string enclosed in ```json...```
    and parses it into a Python dict or list.

    Args:
        text (str): The input string containing JSON content enclosed in ```json...```.

    Returns:
        dict or list: Parsed JSON object (Python dict or list).
    """
    # Use regex to find JSON block
    match = re.search(r'```json\s(.*?)```', text, re.DOTALL)

    if match:
        # Extract JSON content
        json_content = match.group(1).strip()

        try:
            return json.loads(json_content)
        except json.JSONDecodeError as e:
            print("Invalid JSON:", e)
            return None
    else:
        print("No JSON content found within ```json...``` delimiters.")
        return None

In [23]:
dict_output = json_string_to_dict(ai_msg_json.content)
dict_output

{'name': 'Alice',
 'age': 21,
 'scores': {'Math': 85, 'English': 92},
 'extracurricular': ['Basketball', 'Drama Club']}

In [24]:
dict_output['scores']['Math']

85

### Output : Pydantic Schema

- LangChain supports structured outputs, **allowing us to bind a schema (dict / JSON Schema / Pydantic) to the model**
  - and enforce responses to follow the defined schema (data type) instead of relying only on prompt wording.
- ***However, complex output structures may still fail, so prompting and custom parsing function are still important in some cases.***

Read more: [LangChain Docs – Structured Outputs](https://python.langchain.com/docs/concepts/structured_outputs/)


In [25]:
# pydantic schema

# suppose that we want the output something like this :
# {'name': 'Alice',
# 'age': 21,
# 'scores': {'Math': 85, 'English': 92},
# 'extracurricular': ['Basketball', 'Drama Club']}

# we can defined class (data fields) like this

from typing import Dict, List
from pydantic import BaseModel, Field

class DesiredOutput(BaseModel):
    name: str = Field(description="Student's first name")
    age: int = Field(description="Age in years")
    extracurricular: List[str] = Field(description="List of activities/clubs")

    #subject_scores: Dict[str, int] = Field(description="Key = subject, Value = scores (as a JSON object)") # This line cause an error. / Complex Data Structure (uncomment if you want to test it)

In [26]:
# Wrap LLM so it returns a DesiredOutput object directly
structured_llm = llm.with_structured_output(DesiredOutput)

In [27]:
prompt = """
You are a helpful assistant.
For the given student record, extract informations

Student Record:
Alice, 21 years old. Math = 85, English = 92. She joined Basketball and Drama Club.

Answer:
"""


# Generate output
results = structured_llm.invoke(prompt)
results

DesiredOutput(name='Alice', age=21, extracurricular=['Basketball', 'Drama Club'])

In [28]:
results.model_dump_json()

'{"name":"Alice","age":21,"extracurricular":["Basketball","Drama Club"]}'

## Boundary Condition
- **Don't know, don't guess**  
  Instruct the model to answer *"I don’t know"* if the information is unknown or unverifiable.  
  → Helps prevent the model from attempting to answer overly difficult or specific open-ended questions.  
  > Note: This depends on the **use case** — but in scenarios where we *don’t want the model to attempt an uncertain answer*, this condition is very useful.

- **Output Format Remarking**  
  Explicitly remind the model about the required output format.  
  → e.g., *"Don’t give any additional explanation, just output [format] only."*

In [29]:
# Example 1: Without boundary condition
prompt = """
What are the details of the announcement from the Meteorological Department, Announcement No. 2/2025, regarding ‘Measures to Cope with the Early Arrival of Summer Storms’?
"""

ai_msg = llm.invoke(prompt)
print("Without boundary condition:\n")
display(Markdown(ai_msg.content))

Without boundary condition:



Based on current available information, **there is no official public record of a "Meteorological Department Announcement No. 2/2025" regarding ‘Measures to Cope with the Early Arrival of Summer Storms’**.

Here’s why:

1.  **Date Context**: The current date is **2024**. Announcements for **2025** have not yet been issued by any national meteorological department, as the year has not occurred.
2.  **No Known Pre-issued Announcements**: Meteorological departments (such as India’s IMD, US’s NWS, UK’s Met Office, etc.) do not typically issue numbered annual announcements years in advance for seasonal weather measures. Guidelines are usually updated seasonally or as needed based on real-time climate forecasts.
3.  **Possible Confusion**:
    - You may be referring to a **past announcement** (e.g., from 2023 or 2024) with a similar title.
    - It could be a **hypothetical**, **fictional**, or **misremembered** reference.
    - There might be confusion with a **different document** (e.g., a school notice, local municipal advisory, or non-governmental report).

### Recommended Actions:
- **Verify the Year**: Confirm if you meant **2024** or an earlier year.
- **Check the Source**: Identify the specific

**Key Takeaways**:
- Without clear boundary conditions, an LLM will still attempt to generate an **answer—sometimes hallucinating content** **(especially in smaller models), and other times making an educated guess while acknowledging its uncertainty.**
- **Define clear boundary conditions and fallback responses so that uncertain cases can be reliably detected and handled in an automated pipeline.**
- This keeps your system consistent and predictable.

In [30]:
# Example 2: With boundary condition
prompt = """
What are the details of the announcement from the Meteorological Department, Announcement No. 2/2025, regarding ‘Measures to Cope with the Early Arrival of Summer Storms’?

If the answer is not known or cannot be verified, just reply: `None`.
"""

ai_msg = llm.invoke(prompt)
print("With boundary condition:\n", ai_msg.content)


With boundary condition:
 None


## Prompt Template

Prompt templates offer several benefits:

- **Consistency**: Ensure a consistent structure for your prompts across multiple interactions
- **Efficiency**: Easily swap out variable content without rewriting the entire prompt
- **Testability**: Quickly test different inputs and edge cases by changing only the variable portion
- **Scalability***: Simplify prompt management as your application grows in complexity
- **Version control**: Easily track changes to your prompt structure over time by keeping tabs only on the core part of your prompt, separate from dynamic inputs

### Example: Prompt Template in a Loop (Task: Sentiment Analysis)

Example Task: **Sentiment Analysis**

We used a prompt template with the approach **“run in a loop + change only variables”**.  
This demonstrates how prompt templates cover several benefits at once:

- **Consistency**: Every iteration uses the same prompt structure.  
- **Efficiency**: Only the variable `{text}` changes in each loop.  
- **Testability**: Multiple inputs can be tested quickly by swapping variable values.  
- **Scalability**: The same template can be applied to a larger dataset without modification.  
- **Version Control**: Easily track prompt versions against results.




In [31]:
!wget https://github.com/neubig/anlp-code/raw/refs/heads/main/data/sst-sentiment-text-threeclass/dev.txt

--2026-09-11 05:31:13--  https://github.com/neubig/anlp-code/raw/refs/heads/main/data/sst-sentiment-text-threeclass/dev.txt
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/neubig/anlp-code/refs/heads/main/data/sst-sentiment-text-threeclass/dev.txt [following]
--2026-09-11 05:31:13--  https://raw.githubusercontent.com/neubig/anlp-code/refs/heads/main/data/sst-sentiment-text-threeclass/dev.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 122071 (119K) [text/plain]
Saving to: ‘dev.txt’

dev.txt             100%[===================>] 119.21K  --.-KB/s    in 0.01s   

2026-09-11 05:31:14 (9.70 MB/

In [32]:
def read_xy_data(filename: str) -> tuple[list[str], list[int]]:
    x_data = []
    y_data = []
    with open(filename, 'r') as f:
        for line in f:
            label, text = line.strip().split(' ||| ')
            x_data.append(text)
            y_data.append(int(label))
    return x_data, y_data

In [33]:
x_test, y_test = read_xy_data('dev.txt')
x_test, y_test = x_test[:3], y_test[:3] # small size, respect the rate limit

For sentiment analysis, we will be using the following prompt:

```
Analyse the sentiment of the following text: ```text```
if the sentiment is positive output '1', '0' for neutral, and '-1' for negative.
**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**
```
LLMs nowaday usually have chain-of-thought baked in so they usually will output their reasoning before answering.

- It is important to tell the model not to output their explanation by including `**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**
`
- Otherwise, it will not be easy to programmatically use the outputs.
Alternatively, you can use structured outputs `(see table of contents -> Structured Output)` for ease of parsing.

In [34]:
prompt_template = """
Analyse the sentiment of the following text: ```{x_input}```
if the sentiment is positive output '1', '0' for neutral, and '-1' for negative.
**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**"""

In [35]:
import time
from tqdm.notebook import tqdm

output = []

# Practical use: add try/except for automatic retries,
# exponential backoff to handle temporary API/rate-limit errors,
# and sleep between requests to respect the provider's rate limits.

max_retries = 5
for sent in tqdm(x_test):
    prompt_filled = prompt_template.format(x_input=sent)
    print("prompt:", prompt_filled)  # debugging
    for attempt in range(max_retries):
        try:
            output_res = llm.invoke(prompt_filled).content.strip()
            print("response:", output_res)
            print("--" * 20)
            output.append(int(output_res))
            time.sleep(3)
            break

        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt # exponential backoff
                print(
                    f"Request failed: {e}\n"
                    f"Retrying in {wait_time} seconds..."
                )
                time.sleep(wait_time)
            else:
                print(f"Failed after {max_retries} attempts.")
                output.append(0)

  0%|          | 0/3 [00:00<?, ?it/s]

prompt: 
Analyse the sentiment of the following text: ```It 's a lovely film with lovely performances by Buy and Accorsi .```
if the sentiment is positive output '1', '0' for neutral, and '-1' for negative.
**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**
response: 1
----------------------------------------
prompt: 
Analyse the sentiment of the following text: ```No one goes unindicted here , which is probably for the best .```
if the sentiment is positive output '1', '0' for neutral, and '-1' for negative.
**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**
response: 1
----------------------------------------
prompt: 
Analyse the sentiment of the following text: ```And if you 're not nearly moved to tears by a couple of scenes , you 've got ice water in your veins .```
if the sentiment is positive output '1', '0' for neutral, and '-1' for negative.
**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**
response: 1
----------------------------------------


In [36]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, output)

0.6666666666666666

## Additional: Temperature Setting

Temperature is a parameter that controls the randomness and diversity of an LLM’s output.
  - Keep it low if you are looking for more consistent and deterministic responses across repeated runs
  - Keep it high if you are looking for more diverse or creative responses.

### Approach 1 : Gemini

Temperature Range for Gemini-2.5-flash : 0-2 (default 1)

>Ref: https://cloud.google.com/vertex-ai/generative-ai/docs/models/gemini/2-5-flash

In [37]:
# from langchain_google_genai import ChatGoogleGenerativeAI

# llm_low_temp = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=0,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
#     # other params...
# )

In [38]:
# llm_high_temp = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=2,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
#     # other params...
# )

In [39]:
# # llm_low_temp
# prompt = "Write one slogan for a mobile banking application. Just only answer the slogan without additional suggestions"
# for rnd in range(3):
#   try:
#     output_res = llm_low_temp.invoke(prompt).content.strip()
#     print(f"Round {rnd+1} | response:", output_res)
#   except Exception as e:
#     print(f"Round {rnd+1} | Error:", e)

#   print("--"*30)
#   time.sleep(2)

In [40]:
# # llm_high_temp
# prompt = "Write one slogan for a mobile banking application. Just only answer the slogan without additional suggestions"
# for rnd in range(3):
#   try:
#     output_res = llm_high_temp.invoke(prompt).content.strip()
#     print(f"Round {rnd+1} | response:", output_res)
#   except Exception as e:
#     print(f"Round {rnd+1} | Error:", e)

#   print("--"*30)
#   time.sleep(2)

### Approach 2 : Groq

In [41]:
llm_low_temp = ChatGroq(
    model="qwen/qwen3.6-27b", # can change
    temperature=0,
    reasoning_effort="none",
    max_tokens=300,
    timeout=None,
    max_retries=2,
)

In [42]:
llm_high_temp = ChatGroq(
    model="qwen/qwen3.6-27b", # can change
    temperature=0.9,
    reasoning_effort="none",
    max_tokens=300,
    timeout=None,
    max_retries=2,
)

**Low-temperature LLM :**

In [43]:
prompt = "Write one slogan for a mobile banking application. Just only answer the slogan without additional suggestions"
for rnd in range(3):
  try:
    output_res = llm_low_temp.invoke(prompt).content.strip()
    print(f"Round {rnd+1} | response:", output_res)
  except Exception as e:
    print(f"Round {rnd+1} | Error:", e)

  print("--"*30)
  time.sleep(3.5)

Round 1 | response: Banking at your fingertips.
------------------------------------------------------------
Round 2 | response: Banking at your fingertips.
------------------------------------------------------------
Round 3 | response: Banking at your fingertips.
------------------------------------------------------------


**High-temperature LLM: **

In [44]:
# llm_high_temp
prompt = "Write one slogan for a mobile banking application. Just only answer the slogan without additional suggestions"
for rnd in range(3):
  try:
    output_res = llm_high_temp.invoke(prompt).content.strip()
    print(f"Round {rnd+1} | response:", output_res)
  except Exception as e:
    print(f"Round {rnd+1} | Error:", e)

  print("--"*30)
  time.sleep(2) # Adding a 2-second delay to avoid rate limit error

Round 1 | response: Banking made simple, anytime, anywhere.
------------------------------------------------------------
Round 2 | response: Banking at your fingertips.
------------------------------------------------------------
Round 3 | response: Banking in your pocket, always open.
------------------------------------------------------------


Summary

- Low temp → Reliable, consistent outputs. Useful for classification, extraction, or when you want reproducibility.
- High temp → Diverse, creative slogans. Useful for brainstorming, ideation, or when multiple fresh options are desired.

## Additional: Reasoning Effort Setting

Nowaday models support `thinking` mode:
- reasoning_effort="none" → faster, lower token usage, suitable for simple tasks
- reasoning_effort="default" → enables reasoning, useful for tasks that require multi-step thinking

In [45]:
prompt = """
A startup has two options for launching a new AI feature:

Option A:
- Faster to build
- Lower development cost
- Uses a less accurate model
- Can launch in 2 weeks

Option B:
- Higher development cost
- More accurate and reliable
- Requires 6 weeks to launch
- Better suited for long-term scaling

The company has limited budget but wants to build user trust.
Which option would you recommend, and why?

Answer in no more than 120 words.
"""

**Fast (No Reasoning)**

In [46]:
llm_fast = ChatGroq(
    model="qwen/qwen3.6-27b",
    reasoning_effort="none",
    max_tokens=300
)

# No reasoning
start = time.perf_counter()
res_fast = llm_fast.invoke(prompt)
latency_fast = time.perf_counter() - start

display(Markdown("### No Reasoning"))
display(Markdown(res_fast.content))
print(f"Latency: {latency_fast:.2f} seconds")

### No Reasoning

I recommend **Option A**, but with a critical caveat: treat it as a Minimum Viable Product (MVP) to validate market fit and gather user feedback quickly. Given the limited budget, spending heavily on Option B’s development without confirmed demand is risky. Launching in two weeks allows the startup to generate early revenue and cash flow to fund future improvements.

However, since user trust is a priority, the team must be transparent about the feature’s limitations and implement robust safeguards to prevent major errors. Once market validation is secured and funds are generated, the company should immediately pivot to building Option B’s superior accuracy and scalability. This iterative approach balances financial constraints with long-term reliability, ensuring the product evolves based on real user needs rather than assumptions.

Latency: 0.46 seconds


**Longer (Reasoning)**

In [51]:
llm_reasoning = ChatGroq(
    model="qwen/qwen3.6-27b",
    reasoning_effort="default",
    max_tokens=300
)

# No reasoning
start = time.perf_counter()
res_reasoning = llm_reasoning.invoke(prompt)
latency_reasoning = time.perf_counter() - start

display(Markdown(res_reasoning.content))
print(f"Latency: {latency_reasoning:.2f} seconds")


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Context:** Startup launching a new AI feature
   - **Option A:** Faster (2 weeks), lower cost, less accurate model
   - **Option B:** Higher cost, more accurate/reliable, longer (6 weeks), better for long-term scaling
   - **Constraints:** Limited budget, wants to build user trust
   - **Task:** Recommend an option and explain why
   - **Constraint:** Max 120 words

2.  **Identify Key Trade-offs:**
   - Budget vs. Quality/Trust
   - Speed vs. Reliability/Scaling
   - User trust is explicitly mentioned as a goal
   - Limited budget is a constraint

3.  **Evaluate Options against Constraints:**
   - Option A fits budget and speed but compromises on accuracy, which directly undermines user trust in an AI feature.
   - Option B fits the trust goal and long-term scaling, but strains the limited budget and delays launch.
   - Need a balanced recommendation that addresses both constraints while prioritizing the stated goal (user trust).

4.  **Formulate Recommendation:**
   - Recommend Option B, but suggest a phased approach to mitigate budget constraints.
   - Why? User trust in AI heavily depends on reliability and accuracy. A flawed launch can damage reputation irreparably and increase long-term costs from support, churn, and rework

Latency: 0.83 seconds
